# Sprint 1 — Dataset Exploration & Understanding

## Objective

Before building any deep learning model, it is essential to understand the dataset.

This notebook performs a comprehensive exploratory analysis of the Mushroom Species dataset.

The goals are:

- Understand the dataset structure
- Validate image paths
- Analyze class distribution
- Measure dataset balance
- Analyze image resolutions
- Detect corrupted or missing images
- Perform initial quality assessment

The observations obtained from this notebook will guide preprocessing strategies, model selection, augmentation policies and future experiments.

## 2. Imports

In [ ]:
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt

plt.style.use("ggplot")

## 3. Dataset Configuration

In [ ]:
DATASET_ROOT = Path("datasets/extracted/mushroom")

TRAIN_CSV = DATASET_ROOT / "train.csv"
VAL_CSV = DATASET_ROOT / "val.csv"
TEST_CSV = DATASET_ROOT / "test.csv"

IMAGE_ROOT = DATASET_ROOT / "merged_dataset"

## 4. Load Dataset

### Dataset Loading

The predefined CSV files are loaded into memory.

These CSV files define the official train, validation and test splits provided by the dataset creators.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"Train      : {train_df.shape}")
print(f"Validation : {val_df.shape}")
print(f"Test       : {test_df.shape}")
train_df.head()

## 5. Dataset Overview

This section summarizes the overall size of the dataset and the number of unique mushroom species.

In [ ]:
total_images = len(train_df) + len(val_df) + len(test_df)

num_classes = train_df["label"].nunique()

print(f"Total Images : {total_images:,}")
print(f"Number of Classes : {num_classes}")

## 6. Dataset Statistics

We compute descriptive statistics to understand how images are distributed among species.

In [ ]:
class_counts = train_df["label"].value_counts()

display(class_counts.describe())
print("Minimum :", class_counts.min())
print("Maximum :", class_counts.max())
print("Mean :", class_counts.mean())
print("Median :", class_counts.median())
print("Std :", class_counts.std())

## 7. Class Distribution

A balanced class distribution generally leads to more stable training and unbiased evaluation.

In [ ]:
plt.figure(figsize=(18,6))

class_counts.plot(kind="bar")

plt.xlabel("Species")
plt.ylabel("Images")
plt.title("Class Distribution")

plt.tight_layout()

plt.show()
display(class_counts.head(10))
display(class_counts.tail(10))

## 8. Image Path Validation

The dataset was originally prepared inside Kaggle.

Therefore, image paths stored inside the CSV files must be converted into local project paths before training.
İlk burada küçük bir mühendislik yapacağız.

In [ ]:
train_df["local_path"] = train_df["image_path"].str.replace(
    "/kaggle/working/merged_dataset",
    str(IMAGE_ROOT),
    regex=False,
)

val_df["local_path"] = val_df["image_path"].str.replace(
    "/kaggle/working/merged_dataset",
    str(IMAGE_ROOT),
    regex=False,
)

test_df["local_path"] = test_df["image_path"].str.replace(
    "/kaggle/working/merged_dataset",
    str(IMAGE_ROOT),
    regex=False,
)

print("Kontrol edelim.")

train_df[["image_path","local_path"]].head()

print("Dosyalar gerçekten var mı?")

exists = train_df["local_path"].apply(lambda x: Path(x).exists())

print(exists.value_counts())

## 9. Image Resolution Analysis

Image size analysis helps determine an appropriate preprocessing strategy and model input resolution.

In [ ]:
sizes = []

sample_paths = random.sample(train_df["local_path"].tolist(), 1000)

for path in sample_paths:
    img = Image.open(path)
    sizes.append(img.size)
widths = [w for w,h in sizes]
heights = [h for w,h in sizes]
plt.figure(figsize=(8,5))

plt.hist(widths,bins=30)

plt.xlabel("Width")

plt.ylabel("Frequency")

plt.title("Image Width Distribution")

plt.show()
plt.figure(figsize=(8,5))

plt.hist(heights,bins=30)

plt.xlabel("Height")

plt.ylabel("Frequency")

plt.title("Image Height Distribution")

plt.show()
print("Average Width :",np.mean(widths))
print("Average Height :",np.mean(heights))

## 10. Sample Visualization

Random visual inspection helps identify background complexity, image quality and fine-grained similarities between species.

In [ ]:
fig,axes = plt.subplots(4,5,figsize=(15,12))

samples=train_df.sample(20)

for ax,(_,row) in zip(axes.flatten(),samples.iterrows()):

    img=Image.open(row["local_path"])

    ax.imshow(img)

    ax.set_title(row["label"][:15],fontsize=8)

    ax.axis("off")

plt.tight_layout()

plt.show()

## 11. Dataset Integrity Checks

Before model training, we verify that all referenced images exist and can be opened successfully.

In [ ]:
broken=[]

for path in random.sample(train_df["local_path"].tolist(),3000):

    try:
        Image.open(path)

    except Exception:

        broken.append(path)

print("Broken Images :",len(broken))

print("Eksik dosya var mı?")

missing=[]

for path in train_df["local_path"]:

    if not Path(path).exists():

        missing.append(path)

print("Missing Files :",len(missing))

## 12. Initial Findings

Summarize the observations after completing the exploratory analysis.

Questions to answer:

- Is the dataset balanced?
- Are all images accessible?
- Are image resolutions consistent?
- Does the dataset appear suitable for fine-grained classification?
- What preprocessing steps seem necessary?

These findings will directly influence the design of the data pipeline in Sprint 2.

(Bunu sonuçlara göre birlikte dolduracağız.)

## 13. Next Steps

Sprint 2 will focus on building a reusable and modular data pipeline.

Planned tasks include:

- Dataset class implementation
- Path management
- Configuration system
- Albumentations pipeline
- DataLoader
- Batch visualization
- Pipeline validation